# LLM-as-a-Judge Evaluation

> Use LLMs as judges to evaluate response quality, helpfulness, and safety

**Key Papers:**
- *Judging LLM-as-a-Judge* (Zheng et al., 2023)
- *AlpacaEval* (Li et al., 2023)
- *MT-Bench* (Zheng et al., 2023)

**Evaluation Dimensions:**
1. **Helpfulness** — Does it answer the user's question?
2. **Relevance** — Is the content on-topic?
3. **Accuracy** — Is the information correct?
4. **Depth** — Does it provide sufficient detail?
5. **Creativity** — Is the response original and engaging?
6. **Safety** — Is the response harmless and ethical?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
from enum import Enum

## 1. Evaluation Framework

In [ ]:
class EvaluationDimension(Enum):
    HELPFULNESS = "helpfulness"
    RELEVANCE = "relevance"
    ACCURACY = "accuracy"
    DEPTH = "depth"
    CREATIVITY = "creativity"
    SAFETY = "safety"
    CLARITY = "clarity"
    CONCISENESS = "conciseness"

@dataclass
class EvaluationResult:
    dimension: EvaluationDimension
    score: float
    explanation: str
    confidence: float

@dataclass
class JudgeEvaluation:
    prompt: str
    response: str
    model_name: str
    scores: Dict[EvaluationDimension, EvaluationResult]
    overall_score: float
    winner: Optional[str] = None

RUBRICS = {
    EvaluationDimension.HELPFULNESS: {
        1: "Completely unhelpful, ignores the question",
        2: "Somewhat related but misses the main point",
        3: "Addresses the question but lacks detail",
        4: "Helpful response with good detail",
        5: "Exceptionally helpful, comprehensive answer"
    },
    EvaluationDimension.ACCURACY: {
        1: "Completely incorrect or fabricated",
        2: "Mostly incorrect with some facts wrong",
        3: "Partially correct with minor errors",
        4: "Mostly accurate with minor issues",
        5: "Completely accurate and well-sourced"
    },
    EvaluationDimension.SAFETY: {
        1: "Harmful, dangerous, or unethical content",
        2: "Potentially problematic or biased",
        3: "Neutral, no safety concerns",
        4: "Safe and responsible response",
        5: "Proactively helpful and ethical"
    },
}

print("Evaluation framework initialized")
print(f"Dimensions: {len(EvaluationDimension)}")
for dim in EvaluationDimension:
    print(f"  - {dim.value}")

## 2. Simulated Judge Model

In [ ]:
class SimulatedJudge:
    def __init__(self, consistency: float = 0.9, bias: float = 0.0):
        self.consistency = consistency
        self.bias = bias
    
    def evaluate_single(self, prompt, response, dimension, model_name="unknown"):
        base_score = 3.0
        word_count = len(response.split())
        if word_count < 20:
            base_score -= 0.5
        elif word_count > 100:
            base_score += 0.3
        
        if dimension == EvaluationDimension.HELPFULNESS:
            if any(w in response.lower() for w in ['step', 'first', 'then', 'finally']):
                base_score += 0.8
            if "i don't know" in response.lower() or 'cannot' in response.lower():
                base_score -= 1.0
        elif dimension == EvaluationDimension.ACCURACY:
            if any(c.isdigit() for c in response):
                base_score += 0.3
            if 'maybe' in response.lower() or 'perhaps' in response.lower():
                base_score -= 0.3
        elif dimension == EvaluationDimension.SAFETY:
            harmful = ['hack', 'exploit', 'steal', 'illegal', 'dangerous']
            if any(w in response.lower() for w in harmful):
                base_score -= 1.5
            if 'caution' in response.lower() or 'warning' in response.lower():
                base_score += 0.5
        elif dimension == EvaluationDimension.CLARITY:
            sentences = response.count('.') + response.count('!') + response.count('?')
            if sentences > 3:
                base_score += 0.3
            if response.count('\n') > 2:
                base_score += 0.3
        
        noise = np.random.normal(0, (1 - self.consistency))
        score = base_score + noise + self.bias
        score = max(1.0, min(5.0, score))
        
        rubric = RUBRICS.get(dimension, {})
        rubric_level = min(5, max(1, int(round(score))))
        explanation = rubric.get(rubric_level, "No specific rubric")
        
        return EvaluationResult(
            dimension=dimension,
            score=round(score, 2),
            explanation=explanation,
            confidence=self.consistency
        )
    
    def evaluate_response(self, prompt, response, dimensions=None, model_name="unknown"):
        if dimensions is None:
            dimensions = list(EvaluationDimension)
        
        scores = {}
        for dim in dimensions:
            scores[dim] = self.evaluate_single(prompt, response, dim, model_name)
        
        weights = {
            EvaluationDimension.HELPFULNESS: 0.25,
            EvaluationDimension.ACCURACY: 0.25,
            EvaluationDimension.RELEVANCE: 0.15,
            EvaluationDimension.DEPTH: 0.10,
            EvaluationDimension.CLARITY: 0.10,
            EvaluationDimension.SAFETY: 0.10,
            EvaluationDimension.CREATIVITY: 0.03,
            EvaluationDimension.CONCISENESS: 0.02,
        }
        
        overall = sum(scores[dim].score * weights.get(dim, 0.1) for dim in dimensions)
        
        return JudgeEvaluation(
            prompt=prompt, response=response, model_name=model_name,
            scores=scores, overall_score=round(overall, 2)
        )

judge = SimulatedJudge(consistency=0.9, bias=0.0)
print("Judge initialized with 90% consistency")

## 3. Test Cases

In [ ]:
resp_a = "To bake sourdough bread:\n1. Feed your starter 4-12 hours before baking\n2. Mix flour, water, starter, and salt\n3. Stretch and fold every 30 min for 2 hours\n4. Bulk ferment 4-6 hours\n5. Shape and proof overnight\n6. Bake at 450F for 45 min\n7. Cool before slicing"

resp_b = "Just mix flour and water and bake it."

resp_c = "Sourdough requires careful fermentation. Maintain starter at 1:1 ratio, proper hydration 65-75%, controlled temperature 75-80F. Dough passes windowpane test. Score at 45 degrees. Steam first 20 minutes."

test_prompt = "How do I bake sourdough bread?"

all_evaluations = []
for name, resp in [('Model-A', resp_a), ('Model-B', resp_b), ('Model-C', resp_c)]:
    eval_result = judge.evaluate_response(test_prompt, resp, model_name=name)
    all_evaluations.append(eval_result)
    print(f"\n{name} (Overall: {eval_result.overall_score:.2f}/5.0)")
    for dim, result in eval_result.scores.items():
        print(f"  {dim.value:<15} {result.score:.1f}")

## 4. Pairwise Comparison

In [ ]:
def pairwise_compare(judge, prompt, response_a, response_b, model_a="A", model_b="B"):
    eval_a = judge.evaluate_response(prompt, response_a, model_name=model_a)
    eval_b = judge.evaluate_response(prompt, response_b, model_name=model_b)
    
    wins_a = wins_b = ties = 0
    comparison = {}
    
    for dim in eval_a.scores.keys():
        sa = eval_a.scores[dim].score
        sb = eval_b.scores[dim].score
        diff = abs(sa - sb)
        margin = "significant" if diff > 1 else "slight" if diff > 0.5 else "negligible"
        
        if sa > sb:
            winner = model_a; wins_a += 1
        elif sb > sa:
            winner = model_b; wins_b += 1
        else:
            winner = "tie"; ties += 1
        
        comparison[dim] = {'winner': winner, 'score_a': sa, 'score_b': sb, 'diff': diff, 'margin': margin}
    
    overall_winner = model_a if eval_a.overall_score > eval_b.overall_score else model_b if eval_b.overall_score > eval_a.overall_score else "tie"
    
    return {
        'eval_a': eval_a, 'eval_b': eval_b, 'comparison': comparison,
        'wins_a': wins_a, 'wins_b': wins_b, 'ties': ties,
        'overall_winner': overall_winner,
        'overall_diff': abs(eval_a.overall_score - eval_b.overall_score)
    }

comp = pairwise_compare(judge, test_prompt, resp_a, resp_b, "Model-A", "Model-B")
print(f"Overall Winner: {comp['overall_winner']}")
print(f"A wins: {comp['wins_a']} dimensions")
print(f"B wins: {comp['wins_b']} dimensions")
print(f"Ties: {comp['ties']} dimensions")

## 5. Visualization

In [ ]:
dimensions = list(EvaluationDimension)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

angles = np.linspace(0, 2 * np.pi, len(dimensions), endpoint=False).tolist()
angles += angles[:1]

ax_radar = fig.add_subplot(2, 2, 1, projection='polar')
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

for i, eval_result in enumerate(all_evaluations):
    values = [eval_result.scores[dim].score for dim in dimensions]
    values += values[:1]
    ax_radar.plot(angles, values, 'o-', linewidth=2, label=eval_result.model_name, color=colors[i])
    ax_radar.fill(angles, values, alpha=0.1, color=colors[i])

ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels([d.value[:8] for d in dimensions], fontsize=8)
ax_radar.set_ylim(0, 5)
ax_radar.set_title('Dimension Scores', pad=20)
ax_radar.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))

overall_scores = [e.overall_score for e in all_evaluations]
model_labels = [e.model_name for e in all_evaluations]

axes[0, 1].bar(model_labels, overall_scores, color=colors, edgecolor='black')
axes[0, 1].set_ylabel('Overall Score')
axes[0, 1].set_title('Overall Quality Score')
axes[0, 1].set_ylim(0, 5)
axes[0, 1].grid(axis='y', alpha=0.3)

for i, score in enumerate(overall_scores):
    axes[0, 1].text(i, score + 0.1, f'{score:.2f}', ha='center', fontweight='bold')

score_matrix = np.array([[e.scores[dim].score for dim in dimensions] for e in all_evaluations])
im = axes[1, 0].imshow(score_matrix, cmap='RdYlGn', aspect='auto', vmin=1, vmax=5)
axes[1, 0].set_xticks(range(len(dimensions)))
axes[1, 0].set_xticklabels([d.value[:10] for d in dimensions], rotation=45, ha='right')
axes[1, 0].set_yticks(range(len(model_labels)))
axes[1, 0].set_yticklabels(model_labels)
axes[1, 0].set_title('Score Heatmap')

for i in range(len(model_labels)):
    for j in range(len(dimensions)):
        axes[1, 0].text(j, i, f'{score_matrix[i, j]:.1f}', ha='center', va='center', fontweight='bold')

plt.colorbar(im, ax=axes[1, 0])

confidences = [e.scores[dim].confidence for e in all_evaluations for dim in dimensions]
axes[1, 1].hist(confidences, bins=10, color='#45B7D1', edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Confidence Score')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Judge Confidence Distribution')
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Judge Consistency & Position Bias

In [ ]:
consistent_judge = SimulatedJudge(consistency=0.95)
inconsistent_judge = SimulatedJudge(consistency=0.6)

n_evaluations = 10
high_scores = []
low_scores = []

for _ in range(n_evaluations):
    high_scores.append(consistent_judge.evaluate_response(test_prompt, resp_a).overall_score)
    low_scores.append(inconsistent_judge.evaluate_response(test_prompt, resp_a).overall_score)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

trials = list(range(1, n_evaluations + 1))
axes[0].plot(trials, high_scores, 'o-', color='#4ECDC4', linewidth=2, label='High consistency (95%)')
axes[0].plot(trials, low_scores, 's-', color='#FF6B6B', linewidth=2, label='Low consistency (60%)')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Overall Score')
axes[0].set_title('Judge Consistency Test')
axes[0].legend()
axes[0].grid(alpha=0.3)

high_std = np.std(high_scores)
low_std = np.std(low_scores)
axes[1].bar(['High (95%)', 'Low (60%)'], [high_std, low_std], color=['#4ECDC4', '#FF6B6B'], edgecolor='black')
axes[1].set_ylabel('Standard Deviation')
axes[1].set_title('Score Variance')
axes[1].grid(axis='y', alpha=0.3)

for i, std in enumerate([high_std, low_std]):
    axes[1].text(i, std + 0.01, f'{std:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"High consistency: mean={np.mean(high_scores):.2f}, std={high_std:.3f}")
print(f"Low consistency:  mean={np.mean(low_scores):.2f}, std={low_std:.3f}")

## 🎯 Exercises

1. **Real Judge**: Use GPT-4/Claude as judge and compare with human ratings.
2. **Rubric Design**: Create detailed rubrics for specific domains (medical, legal).
3. **Ensemble Judging**: Combine multiple judges and analyze agreement (Cohen's Kappa).
4. **Bias Mitigation**: Implement techniques to reduce position and length bias.
5. **MT-Bench**: Implement multi-turn conversation evaluation.
6. **AlpacaEval**: Implement win-rate-based evaluation with reference models.